# Competencia 01 — de cero

Un solo archivo. Se corre de arriba a abajo. Cada bloque deja una tabla en DuckDB y
**verifica** lo que produjo antes de seguir.

Modalidad: DuckDB + SQL con magics `%%sql`, igual que `z101_target_sql` y
`z402_Feature_Engineering_en_SQL`. El SQL hace el trabajo pesado; Python solo orquesta,
genera texto de SQL y mide.

| Bloque | Qué hace | Verifica |
|---|---|---|
| 1 | Entorno, semillas, ganancia | — |
| 2 | Carga del crudo | granularidad y conteos por mes |
| 3 | Target ternario | casos de `BAJA+2` y madurez de cada mes |
| 4 | Controles de fuga | 6 fugas posibles |
| 5 | Función de ganancia | el cálculo en pesos |
| 6 | Fábrica de features | que ninguna lea el futuro |

Lo que **todavía no** está acá: medición con 20 semillas, canaritos y Optuna. Van después,
cuando este archivo esté corrido y verificado.

## 1. Entorno

In [1]:
%pip install duckdb jupysql duckdb-engine lightgbm optuna

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
import duckdb
import numpy as np
import pandas as pd

pd.set_option("display.max_rows", 300)

%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False

In [3]:
%pip install -U jupysql duckdb-engine

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [4]:
# Ruta del crudo. Se prueban varias porque el dataset vive en Drive montado.
from pathlib import Path

rutas_posibles = [
    Path("data/competencia_01_crudo.csv"),
    Path("competencia_01_crudo.csv"),
    Path("/Users/manuelmarasco/Library/CloudStorage/"
         "GoogleDrive-hola@manuelmarasco.com/Mi unidad/DMEyF/2026/"
         "notebooks/data/competencia_01_crudo.csv"),
]
dataset = next(r for r in rutas_posibles if r.is_file())

# La base se persiste en un archivo y no en memoria: si se corta la sesión,
# las tablas ya construidas siguen estando.
base_duckdb = "competencia_01.duckdb"

print("crudo:", dataset)
print("base :", base_duckdb)

crudo: /Users/manuelmarasco/Library/CloudStorage/GoogleDrive-hola@manuelmarasco.com/Mi unidad/DMEyF/2026/notebooks/data/competencia_01_crudo.csv
base : competencia_01.duckdb


In [5]:
# UNA sola conexión para todo el notebook, compartida entre los magics %%sql
# y el código Python.
#
# Por qué importa: DuckDB no permite abrir el mismo archivo dos veces con
# configuraciones distintas en el mismo proceso. Si se deja que %sql abra su
# propia conexión con duckdb:///archivo y después se hace duckdb.connect(),
# la segunda falla con "Can't open a connection to same database file with a
# different configuration". Pasándole a jupysql la conexión ya abierta, los
# dos mundos ven las mismas tablas y no hay conflicto.

In [6]:
con = duckdb.connect(base_duckdb)

%sql con

In [7]:
# --- las cinco semillas declaradas en el CheckIn ---
# Son las únicas que pueden aparecer en la entrega: la cátedra reproduce la
# predicción desde el crudo y tiene que dar bit a bit lo mismo.
semillas = [100129, 100183, 100151, 500041, 900007]

# --- semillas extra, solo para MEDIR ---
# El desvío de ganancia entre semillas en junio es de 13,2 M. Con 5 semillas el
# error estándar de la media es 5,9 M, así que no se puede distinguir un bloque
# que aporta 5 M de uno que aporta 0. Ensamblar satura en 5; medir necesita 20.
semillas_medicion = semillas + [
    103687, 150151, 212777, 265543, 318211,
    371299, 424243, 477551, 530177, 583651,
    636851, 690037, 743251, 796553, 849839,
]

# --- el problema de negocio ---
ganancia_acierto = 1_072_500
costo_estimulo   =    27_500

# Punto de indiferencia: p*ganancia - (1-p)*costo = 0  =>  p = costo/(ganancia+costo)
umbral = costo_estimulo / (ganancia_acierto + costo_estimulo)

print("umbral de indiferencia:", umbral, "= 1 /", round(1/umbral))
print("asimetría:", round(ganancia_acierto/costo_estimulo), "a 1")

umbral de indiferencia: 0.025 = 1 / 40
asimetría: 39 a 1


In [8]:
# --- la ventana temporal ---
mes_scoring = 202108                                    # el mes a predecir
meses_target = [202103, 202104, 202105, 202106]         # los únicos con clase completa
meses_maduracion = 2                                    # BAJA+2 necesita ver M+1 y M+2


def indice_mes(foto_mes):
    """202104 -> entero continuo de meses.

    Restar foto_mes directo funciona dentro del año y falla al cruzarlo:
    202201 - 202112 da 89 en vez de 1. En la Última Competencia, que abarca
    18 meses, eso rompe cualquier verificación de gap en silencio.
    """
    return (foto_mes // 100) * 12 + (foto_mes % 100)


print("202112 -> 202201:", indice_mes(202201) - indice_mes(202112),
      "| resta ingenua:", 202201 - 202112)

202112 -> 202201: 1 | resta ingenua: 89


## 2. Carga del crudo

`sample_size = -1` fuerza a DuckDB a leer **todo** el archivo para inferir los tipos, en
vez de adivinarlos con las primeras 20.000 filas. Sin eso, una columna que es entera
arriba y decimal recién en la fila 500.000 se infiere como entera y DuckDB trunca o
falla. Con 154 columnas de datos bancarios es casi seguro que pasa.

In [9]:
%%sql
create or replace table competencia_01_crudo as
select *
from read_csv_auto('{{dataset}}', sample_size = -1)

,Count
0,983061


In [10]:
%%sql
select foto_mes, count(*) as filas
from competencia_01_crudo
group by foto_mes
order by foto_mes

,foto_mes,filas
0,202103,162900
1,202104,163284
2,202105,163768
3,202106,164114
4,202107,164348
5,202108,164647


In [11]:
# VERIFICACIÓN: el dataset es el que creemos.
# Si la cátedra republica el archivo, esto grita acá y no produce en silencio
# otro resultado tres horas después.
filas_esperadas = {202103: 162_900, 202104: 163_284, 202105: 163_768,
                   202106: 164_114, 202107: 164_348, 202108: 164_647}

conteos = %sql select foto_mes, count(*) as filas from competencia_01_crudo group by foto_mes order by foto_mes
observado = dict(zip(conteos.foto_mes, conteos.filas))

assert observado == filas_esperadas, f"dataset distinto: {observado}"
print("OK: filas por mes coinciden |", f"{sum(observado.values()):,}", "en total")

OK: filas por mes coinciden | 983,061 en total


## 3. El target

### Verificación de granularidad

El par `(numero_de_cliente, foto_mes)` es la clave primaria compuesta. Si se repitiera,
todo lo que viene después se rompe **en silencio**: `lag()` y `lead()` asumen una fila
por cliente por mes, así que con duplicados `lead(presencia, 1)` no devuelve "el mes
siguiente" sino "la otra fila del mismo mes", y el target sale mal sin tirar error.

Cero filas es la respuesta que se busca.

In [12]:
%%sql
select numero_de_cliente, foto_mes, count(*) as n
from competencia_01_crudo
group by numero_de_cliente, foto_mes
having count(*) > 1

,numero_de_cliente,foto_mes,n


In [13]:
duplicados = %sql select count(*) as n from (select numero_de_cliente, foto_mes from competencia_01_crudo group by numero_de_cliente, foto_mes having count(*) > 1)
assert int(duplicados.n.iloc[0]) == 0, "la clave se repite"
print("OK: la clave (numero_de_cliente, foto_mes) no se repite")

OK: la clave (numero_de_cliente, foto_mes) no se repite


### Construcción de `clase_ternaria`

El crudo **no trae la clase**. La baja no es un campo: es el hecho de que el cliente
**deja de aparecer**. La etiqueta del mes M se deduce de la presencia en M+1 y M+2:

| presente en M+1 | presente en M+2 | clase | lectura |
|---|---|---|---|
| no | — | `BAJA+1` | ya se fue, llegamos tarde |
| sí | no | **`BAJA+2`** | **esto es lo que se predice** |
| sí | sí | `CONTINUA` | sigue siendo cliente |

**Por qué el `cross join` y no un `lead()` directo.** Hay 23 clientes que desaparecen y
vuelven. Un `lead()` sobre las filas que *existen* leería mayo como "el mes siguiente de
marzo" para un cliente que faltó en abril, y lo llamaría `CONTINUA` cuando en realidad
desapareció — o sea `BAJA+1` en marzo.

La solución es **materializar la ausencia**: producto cartesiano cliente × mes, marcar
con `mes_0 = 1` dónde el cliente apareció de verdad, y recién entonces aplicar `lead()`.
Así la ventana camina el calendario completo y no la lista de filas existentes. Son 23
clientes en esta ventana; en 18 meses van a ser muchos más.

In [14]:
%%sql
create or replace table competencia_01 as

with periodos as (
    select distinct foto_mes from competencia_01_crudo
),
clientes as (
    select distinct numero_de_cliente from competencia_01_crudo
),
-- una fila por cada (cliente, mes) concebible, exista o no
calendario as (
    select c.numero_de_cliente, p.foto_mes
    from clientes as c cross join periodos as p
),
-- left join: donde el cliente no apareció todo queda null y mes_0 en 0.
-- esa es la ausencia explícita.
presencia as (
    select
        k.numero_de_cliente,
        k.foto_mes,
        d.* exclude (numero_de_cliente, foto_mes),
        case when d.numero_de_cliente is null then 0 else 1 end as mes_0
    from calendario as k
    left join competencia_01_crudo as d
        on  d.numero_de_cliente = k.numero_de_cliente
        and d.foto_mes          = k.foto_mes
),
-- el futuro del cliente, mes a mes.
-- "window w as (...)" se define una vez y se reusa: es la sintaxis de z402
-- para no repetir el over (partition by ...) en cada línea.
etiquetado as (
    select
        *,
        lead(mes_0, 1) over w as presente_en_m1,
        lead(mes_0, 2) over w as presente_en_m2
    from presencia
    window w as (partition by numero_de_cliente order by foto_mes)
)

select
    * exclude (mes_0, presente_en_m1, presente_en_m2),
    case
        when presente_en_m1 = 0                        then 'BAJA+1'
        when presente_en_m1 = 1 and presente_en_m2 = 0 then 'BAJA+2'
        when presente_en_m1 = 1 and presente_en_m2 = 1 then 'CONTINUA'
        -- borde de la ventana: no se puede saber. queda null a propósito,
        -- nunca imputado, para que no entre a entrenar.
        else null
    end as clase_ternaria
from etiquetado
-- solo las filas que existen de verdad. las de ausencia cumplieron su función
-- (hacer que lead() camine el calendario) y se descartan.
where mes_0 = 1
order by foto_mes, numero_de_cliente

,Count
0,983061


In [15]:
%%sql
pivot competencia_01
on clase_ternaria
using count(numero_de_cliente)
group by foto_mes
order by foto_mes

,foto_mes,BAJA+1,BAJA+2,CONTINUA
0,202103,1019,960,160921
1,202104,964,1139,161181
2,202105,1143,870,161755
3,202106,874,1098,162142
4,202107,1103,0,0
5,202108,0,0,0


In [16]:
%%sql
select foto_mes,
       count(*)                                        as filas,
       count(*) filter (clase_ternaria is null)        as sin_clase,
       count(*) filter (clase_ternaria = 'BAJA+1')     as baja1,
       count(*) filter (clase_ternaria = 'BAJA+2')     as baja2,
       count(*) filter (clase_ternaria = 'CONTINUA')   as continua,
       round(100.0 * count(*) filter (clase_ternaria = 'BAJA+2')
             / count(*), 3)                            as tasa_baja2_pct
from competencia_01
group by foto_mes
order by foto_mes

,foto_mes,filas,sin_clase,baja1,baja2,continua,tasa_baja2_pct
0,202103,162900,0,1019,960,160921,0.589
1,202104,163284,0,964,1139,161181,0.698
2,202105,163768,0,1143,870,161755,0.531
3,202106,164114,0,874,1098,162142,0.669
4,202107,164348,163245,1103,0,0,0.000
5,202108,164647,164647,0,0,0,0.000


In [17]:
# VERIFICACIÓN del target, tres cosas distintas.
resumen = %sql select foto_mes, count(*) as filas, count(*) filter (clase_ternaria is null) as sin_clase, count(*) filter (clase_ternaria = 'BAJA+2') as baja2 from competencia_01 group by foto_mes order by foto_mes

# 1. reproduce los casos de BAJA+2 ya validados en el EDA.
baja2_esperados = {202103: 960, 202104: 1_139, 202105: 870, 202106: 1_098}
observado = {int(r.foto_mes): int(r.baja2) for r in resumen.itertuples()
             if int(r.foto_mes) in baja2_esperados}
assert observado == baja2_esperados, f"BAJA+2 no reproduce el EDA: {observado}"

for r in resumen.itertuples():
    mes, sin_clase, filas = int(r.foto_mes), int(r.sin_clase), int(r.filas)
    # 2. los meses maduros no pueden tener nulos.
    if mes in meses_target:
        assert sin_clase == 0, f"{mes} es maduro y tiene {sin_clase} sin clase"
    # 3. el mes de scoring tiene que estar 100% sin etiqueta.
    #    si tuviera alguna, habría fuga de futuro.
    if mes == mes_scoring:
        assert sin_clase == filas, f"{mes} tiene {filas-sin_clase} etiquetadas: FUGA"

print("OK: BAJA+2 reproduce el EDA; maduros completos; 202108 sin etiqueta")

OK: BAJA+2 reproduce el EDA; maduros completos; 202108 sin etiqueta


Dos lecturas de esa tabla:

- **`BAJA+2` es rarísimo: 0,531 % a 0,698 %.** El problema está fuertemente
  desbalanceado, y por eso la métrica no puede ser accuracy.
- **202107 y 202108 están sin etiqueta, y corresponde.** Julio deja ver quién
  desaparece en agosto (eso sería `BAJA+1`) pero no separar `BAJA+2` de `CONTINUA`,
  porque haría falta septiembre. Agosto no tiene nada: es el mes a predecir. Ninguno
  sirve como conjunto supervisado, pero **julio sí aporta historia** para los lags de
  agosto.

## 4. Controles de fuga de información

Seis fugas distintas. No es una sola cosa.

| | Fuga | Cómo se ataja |
|---|---|---|
| **F1** | Ventanas de resultado solapadas: train a 1 mes de la validación | gap ≥ 2 verificado por `assert` |
| **F2** | Etiqueta en el mes de scoring | ya verificado arriba |
| **F3** | Target propagado entre meses del mismo cliente | la clase es del par `(cliente, mes)` |
| **F4** | `numero_de_cliente` o `foto_mes` como predictores | lista de columnas prohibidas |
| **F5** | Feature calculada con datos del futuro | test de truncamiento, bloque 6 |
| **F6** | Fuga de **selección**: elegir y medir en el mismo conjunto | protocolo, no código |

### F1 — El gap de maduración

La etiqueta del mes M codifica la presencia del cliente en M+1 y M+2. Si el último mes
de train está a **1** mes de la validación, las dos etiquetas comparten un mes de
observación: la de train revela parte de la de validación.

Exigiendo distancia ≥ 2 las ventanas quedan disjuntas, **igual que entre 202106** (el
último mes etiquetado, que sí puede entrenar) **y 202108** (el mes a predecir). O sea:
el fold replica la relación temporal real del problema.

Con 6 fotos existen exactamente **dos** folds válidos. No hay más.

In [18]:
folds = [
    {"nombre": "mayo",  "train": [202103],           "validacion": 202105},
    {"nombre": "junio", "train": [202103, 202104],   "validacion": 202106},
]

for f in folds:
    gap = indice_mes(f["validacion"]) - indice_mes(max(f["train"]))
    assert gap >= meses_maduracion, (
        f"fold {f['nombre']}: train hasta {max(f['train'])}, "
        f"validación {f['validacion']}, gap {gap} < {meses_maduracion}")
    print(f"fold {f['nombre']:6s}: train {str(f['train']):20s} -> "
          f"valida {f['validacion']}  gap {gap}  OK")

# El mismo gap que hay en el problema real:
print(f"\nreal         : train hasta 202106       -> predice {mes_scoring}  "
      f"gap {indice_mes(mes_scoring) - indice_mes(202106)}")

fold mayo  : train [202103]             -> valida 202105  gap 2  OK
fold junio : train [202103, 202104]     -> valida 202106  gap 2  OK

real         : train hasta 202106       -> predice 202108  gap 2


### F3 — El target no está propagado

Si alguien copiara la baja de junio hacia todas las filas del cliente, cada cliente
tendría **una sola** clase distinta. Que haya clientes con dos y con tres prueba que la
clase pertenece al par `(cliente, mes)`.

El ID del cliente sirve para **ordenar la historia** y construir lags. No para copiar
etiquetas ni como número predictivo.

In [19]:
%%sql
with t as (
    select numero_de_cliente,
           string_agg(clase_ternaria, ' -> ' order by foto_mes) as trayectoria
    from competencia_01
    where clase_ternaria is not null
    group by numero_de_cliente
)
select trayectoria, count(*) as clientes
from t
group by trayectoria
order by clientes desc
limit 10

,trayectoria,clientes
0,CONTINUA -> CONTINUA -> CONTINUA -> CONTINUA,157837
1,CONTINUA,1475
2,CONTINUA -> CONTINUA,1436
3,CONTINUA -> CONTINUA -> CONTINUA,1384
4,CONTINUA -> BAJA+2 -> BAJA+1,1139
5,CONTINUA -> CONTINUA -> CONTINUA -> BAJA+2 -> ...,1088
6,BAJA+1,1033
7,BAJA+2 -> BAJA+1,964
8,CONTINUA -> CONTINUA -> BAJA+2 -> BAJA+1,869
9,BAJA+2 -> BAJA+1 -> CONTINUA,7


Ahí está la trayectoria que describe la clase: `CONTINUA -> BAJA+2 -> BAJA+1`. El
mismo cliente cambia de clase mes a mes.

### Control de construcción concluyente

Todo `BAJA+2` en el mes M **tiene que ser** `BAJA+1` en M+1. Se deduce de la definición:
`BAJA+2` en M es "presente en M+1, ausente en M+2"; visto desde M+1 eso es exactamente
"ausente el mes que viene", o sea `BAJA+1`.

Si aparece cualquier otra clase, el `lead()` está desalineado. Da 100 %.

In [20]:
%%sql
with pares as (
    select b.clase_ternaria as clase_mes_siguiente
    from competencia_01 a
    join competencia_01 b
      on  b.numero_de_cliente = a.numero_de_cliente
      and (b.foto_mes//100)*12 + (b.foto_mes%100)
        = (a.foto_mes//100)*12 + (a.foto_mes%100) + 1
    where a.clase_ternaria = 'BAJA+2'
)
select clase_mes_siguiente, count(*) as casos
from pares
group by clase_mes_siguiente

,clase_mes_siguiente,casos
0,BAJA+1,4067


In [21]:
control = %sql with pares as (select b.clase_ternaria as c from competencia_01 a join competencia_01 b on b.numero_de_cliente = a.numero_de_cliente and (b.foto_mes//100)*12 + (b.foto_mes%100) = (a.foto_mes//100)*12 + (a.foto_mes%100) + 1 where a.clase_ternaria = 'BAJA+2') select c, count(*) as casos from pares group by c
assert set(control.c) == {"BAJA+1"}, f"target mal construido: {dict(zip(control.c, control.casos))}"
print(f"OK: los {int(control.casos.sum()):,} BAJA+2 con mes siguiente son BAJA+1 el 100%")

OK: los 4,067 BAJA+2 con mes siguiente son BAJA+1 el 100%


### F4 — Columnas prohibidas

`numero_de_cliente` no es una magnitud, es un identificador: si entra, el modelo
memoriza clientes en vez de aprender comportamiento. `foto_mes` es el calendario mismo.

Las columnas de fecha quedan afuera **por ahora** porque son contadores de días contra
una referencia móvil; se convierten a duraciones en el bloque 6.

`*_fue_corregido` son banderas de una corrección técnica, no comportamiento bancario.

In [22]:
columnas_prohibidas = {
    "numero_de_cliente",   # identificador, no magnitud
    "foto_mes",            # el calendario
    "clase_ternaria",      # el target
    "target_baja2",        # el target binarizado
}

# Fechas: contadores de días contra una referencia móvil. Laguearlas no
# significa nada; convertirlas a duración sí (bloque 6).
columnas_fecha = {
    "Master_Fvencimiento", "Master_Finiciomora", "Master_fultimo_cierre",
    "Master_fechaalta", "Visa_Fvencimiento", "Visa_Finiciomora",
    "Visa_fultimo_cierre", "Visa_fechaalta",
}

todas = [r[0] for r in con.execute("describe competencia_01").fetchall()]

features_crudas = [c for c in todas
                   if c not in columnas_prohibidas and c not in columnas_fecha]

assert not (columnas_prohibidas & set(features_crudas)), "hay columnas prohibidas"
print(f"columnas en la tabla   : {len(todas)}")
print(f"features crudas         : {len(features_crudas)}")
print(f"fechas reservadas       : {len(columnas_fecha)}")

columnas en la tabla   : 155
features crudas         : 144
fechas reservadas       : 8


### F6 — La fuga de selección

Esta **no tiene `assert`**, y es la que más cara sale.

Elegir features, hiperparámetros, cantidad de iteraciones o punto de corte mirando el
mismo conjunto donde después se reporta no ensucia los datos: ensucia la **estimación**.

Está medida en este problema: elegir el punto de corte dentro de junio y reportar en
junio infla la ganancia en **+35,88 M**. El corte óptimo de junio, llevado a mayo,
pierde 42,91 M.

Contra eso no hay código, hay protocolo: **el parámetro se elige en un mes y se mide en
el otro**, y la regla de decisión se escribe antes de ver los resultados.

## 5. La función de ganancia

La métrica no es AUC ni accuracy: es **plata**. Estimular a un `BAJA+2` paga
\$1.072.500; estimular a cualquier otro cuesta \$27.500.

Dos cosas que hay que hacer bien:

1. **La ganancia se construye siempre desde la ternaria, nunca desde la binaria con la
   que se entrena.** `BAJA+1` y `CONTINUA` cuestan lo mismo, pero son clientes
   distintos: si se calcula desde la binaria se pierde la capacidad de auditar a quién
   se le estimuló.
2. **El parámetro `prop`**, igual que en `z301`: cuando se mide sobre una muestra del
   30 % de la población, la ganancia se divide por 0,3 para volver a escala completa.

In [23]:
def ganancia_por_fila(clase_ternaria):
    """Lo que paga estimular a cada cliente. Desde la TERNARIA."""
    return np.where(np.asarray(clase_ternaria) == "BAJA+2",
                    ganancia_acierto, -costo_estimulo)


def ganancia(probabilidades, clase_ternaria, prop=1, threshold=umbral):
    """Ganancia de estimular a todos los que superan el umbral."""
    g = ganancia_por_fila(clase_ternaria)
    return g[np.asarray(probabilidades) >= threshold].sum() / prop


def curva_de_ganancia(probabilidades, clase_ternaria):
    """Ganancia acumulada ordenando por probabilidad descendente.

    El primer elemento es 0: la opción de no estimular a nadie. Sin eso, el
    máximo de la curva no puede ser "no hagas la campaña".
    """
    orden = np.argsort(-np.asarray(probabilidades), kind="mergesort")
    return np.concatenate(([0.0], np.cumsum(ganancia_por_fila(clase_ternaria)[orden])))


# Verificación con números a mano: 3 aciertos y 2 errores.
prueba = ganancia([0.9, 0.9, 0.9, 0.9, 0.001],
                  ["BAJA+2", "BAJA+2", "BAJA+2", "CONTINUA", "BAJA+2"])
esperado = 3 * ganancia_acierto - 1 * costo_estimulo
assert prueba == esperado, (prueba, esperado)
print(f"OK: 3 aciertos - 1 error = ${prueba:,.0f}")
print(f"    (el BAJA+2 con probabilidad 0,001 queda sin estimular: no se cobra ni se paga)")

OK: 3 aciertos - 1 error = $3,190,000
    (el BAJA+2 con probabilidad 0,001 queda sin estimular: no se cobra ni se paga)


## 6. La fábrica de features

Acá se cumple lo que pide la clase: **cientos de features temporales**, no `lag1` y
`delta1` sobre un puñado de variables.

El patrón es el de `z402` celdas 27-29: no se escriben a mano, se **genera el texto del
SQL** desde listas de campos. Es lo mismo que hace `dbt`, a mano y en veinte líneas.

### Dos reglas de construcción

**Profundidad máxima: lag 1, 2 y 3.** En 202108 los lags limpios son 202107, 202106 y
202105. Un `lag4` solo existiría para 202107-202108 y generaría el patrón
"nulo ⇒ mes temprano", que el modelo usa como proxy de `foto_mes`. Eso es fuga de
calendario, no señal de churn.

**El delta absoluto arrastra inflación.** Para variables de monto, un delta nominal
mezcla "el cliente gastó menos" con "los precios subieron". Por eso se agrega el delta
**relativo** y el **ratio contra la media móvil**, que se auto-normaliza.

### La parametrización que exige el gate F5

La función recibe el **nombre de la tabla de origen**. Es la razón por la que el gate de
truncamiento puede funcionar: permite reconstruir exactamente las mismas features sobre
una tabla cortada en el mes M y comparar.

In [24]:
# Macros de DuckDB, igual que en z402.
#
# suma_sin_null: sumar con nulos da null, y eso no siempre es lo que se quiere.
# ratio_seguro: devuelve NULL con denominador 0 o nulo, NUNCA 0. LightGBM
#   maneja NA de forma nativa y puede aprender "denominador cero" como
#   categoría propia; un 0 le miente diciendo que el ratio es cero.
con.execute("""
    create or replace macro suma_sin_null(a, b) as
        case when a is null and b is null then null
             else ifnull(a, 0) + ifnull(b, 0) end;
""")
con.execute("""
    create or replace macro ratio_seguro(numerador, denominador) as
        numerador / nullif(denominador, 0);
""")
print("macros suma_sin_null y ratio_seguro creadas")

macros suma_sin_null y ratio_seguro creadas


In [25]:
# --- las variables que reciben tratamiento temporal ---
# No son todas: son las que el informe de features marca como Tier 1, o sea
# cobertura alta, continuas, y donde la caída monótona es la firma del cliente
# que se va. Agrupadas por lectura de negocio para poder discutirlas.

intensidad_de_uso = [
    "ctrx_quarter",                     # la variable canónica de churn
    "ctarjeta_debito_transacciones", "mautoservicio",
    "ctarjeta_visa_transacciones", "mtarjeta_visa_consumo",
    "ctarjeta_master_transacciones", "mtarjeta_master_consumo",
    "chomebanking_transacciones", "cmobile_app_trx",
    "catm_trx", "matm", "catm_trx_other", "matm_other",
    "cextraccion_autoservicio", "mextraccion_autoservicio",
]

saldos_y_fondeo = [          # la plata se va antes que el cliente
    "mcuentas_saldo", "mcaja_ahorro", "mcuenta_corriente",
    "mcaja_ahorro_dolares", "mplazo_fijo_dolares",
    "minversion1_pesos", "minversion2",
]

rentabilidad_y_precio = [
    "mrentabilidad", "mrentabilidad_annual", "mcomisiones",
    "mactivos_margen", "mpasivos_margen",
    "mcomisiones_mantenimiento", "ccomisiones_mantenimiento",
    "mcomisiones_otras",
]

el_ancla = [                 # lo que hace costoso irse
    "cpayroll_trx", "mpayroll",
    "ccuenta_debitos_automaticos", "mcuenta_debitos_automaticos",
    "ctarjeta_visa_debitos_automaticos", "mttarjeta_visa_debitos_automaticos",
    "ctarjeta_master_debitos_automaticos", "mttarjeta_master_debitos_automaticos",
    "cpagomiscuentas", "mpagomiscuentas",
    "cpagodeservicios", "mpagodeservicios",
]

flujos = [
    "ctransferencias_recibidas", "mtransferencias_recibidas",
    "ctransferencias_emitidas", "mtransferencias_emitidas",
    "ccheques_depositados", "mcheques_depositados",
    "ccheques_emitidos", "mcheques_emitidos",
]

tarjetas = [
    "Visa_msaldototal", "Visa_mlimitecompra", "Visa_mconsumototal",
    "Visa_mpagominimo", "Visa_cconsumos",
    "Master_msaldototal", "Master_mlimitecompra", "Master_mconsumototal",
    "Master_mpagominimo", "Master_cconsumos",
]

variables_temporales = (intensidad_de_uso + saldos_y_fondeo
                        + rentabilidad_y_precio + el_ancla + flujos + tarjetas)

# Montos: llevan delta relativo además del absoluto, porque el nominal
# arrastra inflación. Convención del dataset: m* es monto, c* es cantidad.
variables_monto = [v for v in variables_temporales if v.startswith(("m", "Visa_m", "Master_m"))]

assert len(variables_temporales) == len(set(variables_temporales)), "hay repetidas"
faltantes = [v for v in variables_temporales if v not in features_crudas]
assert not faltantes, f"no están en la tabla: {faltantes}"

print(f"variables con tratamiento temporal: {len(variables_temporales)}")
print(f"  de las cuales montos            : {len(variables_monto)}")

variables con tratamiento temporal: 60
  de las cuales montos            : 38


In [26]:
def sql_features(tabla_origen, profundidad_lag=2):
    """Genera el SQL que construye todas las features temporales.

    Recibe la tabla de origen como PARÁMETRO: es lo que le permite al gate F5
    reconstruir las mismas features sobre una tabla truncada y comparar.

    Todas las ventanas son (partition by numero_de_cliente order by foto_mes)
    con marco "rows between N preceding and current row". Dos cosas importan:

    - partition by numero_de_cliente: la historia es de cada cliente, no de la
      tabla. Sin eso, el lag de un cliente trae el valor de otro.
    - el marco "N preceding and current row" mira solo hacia atrás. Un marco
      "following" o una ventana sin marco leerían el futuro: eso es la fuga F5.
    """
    lineas = []

    for v in variables_temporales:
        q = f'"{v}"'
        # Nivel pasado. Hasta lag3 porque en 202108 los lags limpios son
        # 202107, 202106 y 202105. Un lag4 sería proxy de foto_mes.
        for k in range(1, profundidad_lag + 1):
            lineas.append(f"lag({q}, {k}) over w as {v}_lag{k}")
            lineas.append(f"{q} - lag({q}, {k}) over w as {v}_delta{k}")
        # Forma de la caída: media, mínimo, máximo, volatilidad de 3 meses.
        #
        # GUARDA DE PROFUNDIDAD. Sin el "case when count(...) over v3 >= 4",
        # avg() en 202103 promedia UN mes y en 202106 promedia CUATRO: la misma
        # columna significa cosas distintas según el mes, y el modelo la usa
        # como proxy de foto_mes. Exigiendo la ventana completa, los meses sin
        # historia suficiente dan NULL, que es la verdad: no se puede saber.
        profundo = f"count({q}) over v3 >= 4"
        lineas.append(f"case when {profundo} then avg({q}) over v3 end as {v}_media3")
        lineas.append(f"case when {profundo} then min({q}) over v3 end as {v}_min3")
        lineas.append(f"case when {profundo} then max({q}) over v3 end as {v}_max3")
        lineas.append(f"case when {profundo} then stddev_samp({q}) over v3 end "
                      f"as {v}_volatilidad3")
        # regr_slope contra cliente_antiguedad, que avanza de a 1 por mes y
        # hace de reloj. Es el ejemplo de z402 celda 25.
        lineas.append(f"case when {profundo} then "
                      f"regr_slope({q}, cliente_antiguedad) over v3 end "
                      f"as {v}_pendiente3")
        # Contra drift: el rango dentro del mes no depende de la escala ni de
        # la inflación. Es "percent_rank over (partition by foto_mes)" de z402.
        lineas.append(f"percent_rank() over (partition by foto_mes order by {q}) "
                      f"as {v}_rango_mes")
        # Evento: se apagó. Vale más que cien deltas numéricos.
        lineas.append(f"case when lag({q}, 1) over w > 0 and {q} = 0 then 1 "
                      f"when lag({q}, 1) over w is null then null else 0 end "
                      f"as {v}_apago")
        lineas.append(f"case when lag({q}, 1) over w = 0 and {q} > 0 then 1 "
                      f"when lag({q}, 1) over w is null then null else 0 end "
                      f"as {v}_prendio")

    for v in variables_monto:
        q = f'"{v}"'
        # Delta RELATIVO: mata la inflación. ratio_seguro devuelve NULL si el
        # denominador es 0, que es lo correcto: "cambió infinito" no es 0.
        lineas.append(f"ratio_seguro({q} - lag({q}, 1) over w, "
                      f"abs(lag({q}, 1) over w)) as {v}_delta_rel1")
        # Contra la media móvil: se auto-normaliza, mata inflación y escala.
        lineas.append(f"case when count({q}) over v3 >= 4 then "
                      f"ratio_seguro({q}, avg({q}) over v3) end as {v}_vs_media3")

    cuerpo = ",\n    ".join(lineas)

    return f"""
        select
            numero_de_cliente,
            foto_mes,
            {cuerpo}
        from {tabla_origen}
        window
            w  as (partition by numero_de_cliente order by foto_mes),
            v3 as (partition by numero_de_cliente order by foto_mes
                   rows between 3 preceding and current row)
    """


# PROFUNDIDAD DE LAG = 2, no 3. Ver la celda siguiente: lag3 es 100% nulo en
# el train de los DOS folds, así que es una columna muerta que no se puede
# evaluar. Se deja el parámetro para subirlo en la Última Competencia, que
# tiene 18 meses.
PROFUNDIDAD_LAG = 2

sql = sql_features("competencia_01", PROFUNDIDAD_LAG)
print(f"SQL generado: {len(sql.splitlines())} líneas")
print()
print("\n".join(sql.splitlines()[:8]))
print("    ...")

SQL generado: 806 líneas


        select
            numero_de_cliente,
            foto_mes,
            lag("ctrx_quarter", 1) over w as ctrx_quarter_lag1,
    "ctrx_quarter" - lag("ctrx_quarter", 1) over w as ctrx_quarter_delta1,
    lag("ctrx_quarter", 2) over w as ctrx_quarter_lag2,
    "ctrx_quarter" - lag("ctrx_quarter", 2) over w as ctrx_quarter_delta2,
    ...


In [27]:
%%time
# Construcción. DuckDB paraleliza las window functions solo.
con.execute("create or replace table features_temporales as "
            + sql_features("competencia_01", PROFUNDIDAD_LAG))

forma = con.execute("select count(*) from features_temporales").fetchone()[0]
columnas_fe = [r[0] for r in con.execute(
    "describe features_temporales").fetchall()]
features_nuevas = [c for c in columnas_fe if c not in ("numero_de_cliente", "foto_mes")]

print(f"filas: {forma:,} | features nuevas: {len(features_nuevas)}")

filas: 983,061 | features nuevas: 796
CPU times: user 9min 35s, sys: 8min 45s, total: 18min 21s
Wall time: 3min 26s


### Gate F5 — El test de truncamiento

Esto es una **prueba**, no un indicio estadístico.

Si una feature usa solamente información de meses ≤ M, entonces **truncar la tabla en M
no puede cambiarla**. Se construyen las mismas features dos veces —sobre la tabla
completa y sobre la tabla cortada en M— y se comparan fila por fila las filas ≤ M.
Cualquier diferencia es prueba de que la feature leyó el futuro.

Es lo que detecta los cuatro errores de SQL que parecen inocentes:

- `percent_rank() over ()` sin `partition by foto_mes` → rankea contra toda la tabla,
  202108 incluido.
- `avg(x) over (partition by numero_de_cliente)` sin marco → promedia la historia
  completa del cliente, futuro incluido.
- cualquier z-score o min-max calculado globalmente.
- un marco `rows between ... and ... following`.

El `coalesce` con un centinela existe porque en SQL `null != null`: sin eso, dos nulos
se comparan como distintos y todas las features darían diferencia.

In [28]:
%%time
mes_corte = 202106   # se trunca acá y se comparan las filas <= 202106

con.execute(f"create or replace table _truncada as "
            f"select * from competencia_01 where foto_mes <= {mes_corte}")
con.execute("create or replace table _fe_completa as "
            + sql_features("competencia_01", PROFUNDIDAD_LAG))
con.execute("create or replace table _fe_truncada as "
            + sql_features("_truncada", PROFUNDIDAD_LAG))

# Una sola consulta que cuenta, para cada feature, en cuántas filas difiere.
CENTINELA = -987654321.0
comparaciones = ",\n       ".join(
    f'sum(case when coalesce(c."{f}", {CENTINELA}) '
    f'!= coalesce(t."{f}", {CENTINELA}) then 1 else 0 end) as "{f}"'
    for f in features_nuevas)

diferencias = con.execute(f"""
    select {comparaciones}
    from _fe_completa as c
    join _fe_truncada as t
      on t.numero_de_cliente = c.numero_de_cliente
     and t.foto_mes = c.foto_mes
    where c.foto_mes <= {mes_corte}
""").df().T.rename(columns={0: "filas_distintas"})

culpables = diferencias[diferencias.filas_distintas > 0]
con.execute("drop table if exists _truncada")
con.execute("drop table if exists _fe_completa")
con.execute("drop table if exists _fe_truncada")

if len(culpables):
    print(f"FUGA F5: {len(culpables)} features leen el futuro")
    display(culpables)
    raise AssertionError("hay features que leen el futuro")

print(f"OK: las {len(features_nuevas)} features son idénticas al truncar en "
      f"{mes_corte}. Ninguna lee el futuro.")

OK: las 796 features son idénticas al truncar en 202106. Ninguna lee el futuro.
CPU times: user 15min 39s, sys: 6min 42s, total: 22min 21s
Wall time: 4min 9s


### Cuánta historia ve de verdad cada fold

Acá está el límite duro de esta ventana, y no es un bug: es la ventana.

Un `lag3` en el mes 202103 no existe — haría falta 202012. Y los meses de
entrenamiento de los folds son justamente los del **arranque** de la ventana, donde los
lags todavía no existen. El resultado es que el modelo entrena con la columna vacía y
valida con la columna llena.

In [29]:
filas = []
for f in folds + [{"nombre": "ENTREGA", "train": [202103, 202104, 202105, 202106],
                   "validacion": mes_scoring}]:
    ins = ",".join(map(str, f["train"]))
    for etapa, condicion in [("train", f"foto_mes in ({ins})"),
                             ("valida", f"foto_mes = {f['validacion']}")]:
        fila = {"fold": f["nombre"], "etapa": etapa}
        for c in ["ctrx_quarter_lag1", "ctrx_quarter_lag2", "ctrx_quarter_media3"]:
            # NaN no es NULL en SQL: regr_slope y las divisiones devuelven NaN,
            # así que hay que preguntar por los dos para contar bien.
            nulo = con.execute(
                f'select round(100.0 * count(*) filter ("{c}" is null '
                f'or isnan("{c}")) / count(*), 1) from features_temporales '
                f'where {condicion}').fetchone()[0]
            fila[c.replace("ctrx_quarter_", "") + "_%nulo"] = nulo
        filas.append(fila)

pd.DataFrame(filas)

,fold,etapa,lag1_%nulo,lag2_%nulo,media3_%nulo
0,mayo,train,100.0,100.0,100.0
1,mayo,valida,0.9,1.7,100.0
2,junio,train,50.4,100.0,100.0
3,junio,valida,0.9,1.8,2.6
4,ENTREGA,train,25.6,50.8,75.6
5,ENTREGA,valida,0.8,1.5,2.4


Leer esa tabla de arriba a abajo:

- **fold mayo:** entrena con 202103, donde `lag1` es **100 % nulo**. Ese fold **no puede
  evaluar ninguna feature temporal**. La columna está vacía en train y llena en
  validación.
- **fold junio:** entrena con 202103+202104, así que `lag1` es ~50 % nulo (la mitad de
  las filas son de marzo). `lag2` es 100 % nulo en train.
- **entrega:** entrena con 202103-202106, `lag1` queda 25 % nulo y `lag2` 50 %, contra
  ~1-2 % en agosto.

Tres consecuencias prácticas, que son decisiones de modelado y no detalles:

1. **`lag3` se cae.** Es 100 % nulo en el train de los dos folds: LightGBM no la usa
   nunca para un corte, y no hay forma de medir si aporta. Por eso
   `PROFUNDIDAD_LAG = 2`.
2. **La guarda de profundidad era obligatoria.** Sin ella, `media3` promediaba 1 mes en
   202103 y 4 en 202106 — la misma columna con significado distinto según el mes, o sea
   un proxy de `foto_mes` servido en bandeja. Con la guarda, los meses sin ventana
   completa dan NULL, que es la verdad.
3. **Esto explica el experimento de lags que había quedado flojo.** El bloque LAG1 daba
   +5,43 M con p=0,17, y ahora se entiende: la mitad del train no tenía `lag1`. No es
   que los lags no sirvan; es que **esta ventana no los puede evaluar bien.**

En la Última Competencia, con 18 meses, el problema desaparece y `PROFUNDIDAD_LAG` sube.

## 7. Las trampas — ¿el gate F5 realmente funciona?

Un gate que siempre pasa es indistinguible de un gate roto. Así que se le planta fuga a
propósito y se verifica que la detecte.

Dos trampas, que son los dos errores más comunes:

- **`lead()`**: mira el mes siguiente. Fuga descarada.
- **`percent_rank() over ()` sin `partition by foto_mes`**: rankea contra toda la tabla,
  202108 incluido. Parece inocente y es la que de verdad se cuela.

In [30]:
def sql_trampas(tabla_origen):
    """Features con fuga DELIBERADA, para probar el gate."""
    return f"""
        select numero_de_cliente, foto_mes,
            -- TRAMPA 1: lead() mira el futuro directamente.
            lead(ctrx_quarter, 1) over w as trampa_lead,
            -- TRAMPA 2: percent_rank sin partition by foto_mes. El rango de
            -- una fila de marzo depende de todas las filas de agosto.
            percent_rank() over (order by mcuentas_saldo) as trampa_rank_global,
            -- CONTROL: la versión correcta de lo mismo, particionada por mes.
            percent_rank() over (partition by foto_mes order by mcuentas_saldo)
                as control_rank_por_mes
        from {tabla_origen}
        window w as (partition by numero_de_cliente order by foto_mes)
    """


def gate_f5(constructor, columnas, mes_corte=202106):
    """Devuelve las columnas que cambian al truncar, o sea las que leen el futuro."""
    con.execute(f"create or replace table _tr as "
                f"select * from competencia_01 where foto_mes <= {mes_corte}")
    con.execute("create or replace table _a as " + constructor("competencia_01"))
    con.execute("create or replace table _b as " + constructor("_tr"))
    CENT = -987654321.0
    exprs = ",\n ".join(
        f'sum(case when coalesce(a."{c}", {CENT}) != coalesce(b."{c}", {CENT}) '
        f'then 1 else 0 end) as "{c}"' for c in columnas)
    d = con.execute(f"""
        select {exprs} from _a a join _b b
          on b.numero_de_cliente = a.numero_de_cliente and b.foto_mes = a.foto_mes
        where a.foto_mes <= {mes_corte}
    """).df().T.rename(columns={0: "filas_distintas"})
    for t in ("_tr", "_a", "_b"):
        con.execute(f"drop table if exists {t}")
    return d


resultado = gate_f5(sql_trampas,
                    ["trampa_lead", "trampa_rank_global", "control_rank_por_mes"])
resultado["veredicto"] = np.where(resultado.filas_distintas > 0,
                                  "FUGA DETECTADA", "limpia")
print(resultado.to_string())

# El gate tiene que cazar las dos trampas y dejar pasar el control.
assert resultado.loc["trampa_lead", "filas_distintas"] > 0, "no cazó el lead"
assert resultado.loc["trampa_rank_global", "filas_distintas"] > 0, "no cazó el rank global"
assert resultado.loc["control_rank_por_mes", "filas_distintas"] == 0, "falso positivo"
print("\nOK: el gate caza las dos trampas y no da falso positivo en el control.")

                      filas_distintas       veredicto
trampa_lead                  163253.0  FUGA DETECTADA
trampa_rank_global           654065.0  FUGA DETECTADA
control_rank_por_mes              0.0          limpia

OK: el gate caza las dos trampas y no da falso positivo en el control.


## 8. El banco de medición

Esta es la pieza sin la cual no se puede decidir nada.

**El ruido que domina no es el de la muestra, es el de la semilla.** Cuando se comparan
dos juegos de features sobre el mismo mes, el conjunto de clientes es idéntico en los
dos brazos: remuestrearlo no compra resolución. Lo que varía es el árbol. El desvío
entre semillas en junio es **13,2 M**, así que con 5 semillas el error estándar de la
media es 5,9 M y un bloque que aporta 5 M es indistinguible de uno que aporta 0.

Cuatro decisiones del banco, cada una por una razón medida:

1. **Comparación apareada por semilla.** La semilla 100129 del candidato se compara
   contra la semilla 100129 del incumbente, no contra el promedio.
2. **Wilcoxon de rangos signados, de una cola.** No asume normalidad y con n=20 tiene
   poder suficiente para efectos de ~10 M.
3. **Poda secuencial.** Se arranca con 5 semillas; si el candidato ya viene perdiendo
   con evidencia, se corta ahí y no se gastan las 20. Entrenar de más no es gratis.
4. **Tiene que ganar en los DOS folds.** El Experimento 01 mostró que mayo y junio se
   contradicen sistemáticamente. Un brazo que gana en uno solo no se acepta.

Y el cache en disco con **firma por contenido**: la clave incluye una huella de los
valores de cada columna, no solo su nombre. Si se cambia el código de una feature y la
columna sigue llamándose igual, la firma cambia y no se reusa la medición vieja.

In [31]:
import hashlib, json, time
import lightgbm as lgb
from scipy import stats


def huella_columna(valores):
    """Firma barata del CONTENIDO de una columna. Si cambia el código de una
    feature pero no su nombre, la firma cambia y el cache no miente."""
    v = np.asarray(valores, dtype=np.float64)
    fin = v[np.isfinite(v)]
    return f"{len(v)}:{np.isnan(v).sum()}:{fin.sum():.6e}:{(fin**2).sum():.6e}"


def corregir_probabilidad(p, tasa_continua):
    """Deshace el sesgo del undersampling de CONTINUA.

    Si se conserva una fracción t de los negativos, el modelo aprende odds
    inflados en 1/t. Multiplicando los odds por t se vuelve a la escala
    original, y RECIÉN AHÍ tiene sentido cortar en 0,025. Sin esta corrección,
    submuestrear y cortar en 0,025 estimula de más. Es el bug número uno de
    los experimentos de desbalance.
    """
    if tasa_continua >= 1.0:
        return p
    return tasa_continua * p / (tasa_continua * p + 1.0 - p)


def p_wilcoxon(diferencias, alternativa):
    d = np.asarray(diferencias, dtype=float)
    if len(d) == 0 or np.allclose(d, 0):
        return 1.0
    try:
        return float(stats.wilcoxon(d, alternative=alternativa).pvalue)
    except ValueError:
        return 1.0


print("herramientas del banco listas")

herramientas del banco listas


In [32]:
CARPETA_BANCO = Path("banco_medicion")
(CARPETA_BANCO / "pred").mkdir(parents=True, exist_ok=True)
RUTA_CACHE = CARPETA_BANCO / "mediciones.csv"

# Hiperparámetros fijos durante toda la selección de features (ceteris paribus).
# Son los del trial 25. Optuna va DESPUÉS, sobre el set ya fijado.
PARAMS_BASE = {
    "objective": "binary", "metric": "None", "boosting_type": "gbdt",
    "max_depth": 12, "num_leaves": 57, "learning_rate": 0.0144493962,
    "min_data_in_leaf": 1614, "feature_fraction": 0.2613620757,
    "bagging_fraction": 0.5276823218, "bagging_freq": 1,
    "lambda_l1": 6.957248086e-05, "lambda_l2": 4.968141077e-06,
    "min_gain_to_split": 1.5074952598, "max_bin": 31,
    "feature_pre_filter": False, "verbosity": -1,
}

# Iteraciones FIJAS, sin early stopping. Si se eligiera la iteración mirando el
# mes de validación, ese mes dejaría de ser una estimación limpia (fuga F6).
ITERACIONES = 150

# Criterios de decisión, escritos ANTES de medir.
ALPHA = 0.05              # significancia para aceptar
ALPHA_PODA = 0.05         # evidencia para cortar temprano
MEJORA_MINIMA_REL = 0.01  # 1%: por debajo de eso no vale la complejidad
SEMILLAS_PODA = 5         # con cuántas se arranca

print(f"iteraciones fijas: {ITERACIONES} | alpha: {ALPHA} | "
      f"mejora mínima: {MEJORA_MINIMA_REL:.0%}")

iteraciones fijas: 150 | alpha: 0.05 | mejora mínima: 1%


In [33]:
class BancoMedicion:
    """Mide un juego de features en los folds temporales, semilla por semilla.

    Cada medición se escribe a disco apenas termina: si el proceso se corta, al
    relanzar se reusa lo medido y solo se entrena lo que falta (regla 11).
    """

    def __init__(self, base, carpeta=CARPETA_BANCO):
        self.base = base
        self.carpeta = carpeta
        self.ruta = carpeta / "mediciones.csv"
        self.cache = {}
        if self.ruta.is_file():
            for r in pd.read_csv(self.ruta).to_dict("records"):
                self.cache[(r["firma"], r["fold"], int(r["semilla"]))] = r
        self.huellas = {}
        self.clase = base["clase_ternaria"].to_numpy()
        self.y = (self.clase == "BAJA+2").astype(np.int8)
        # Máscaras de train y validación por fold, calculadas una vez.
        self.mascaras = {
            f["nombre"]: (base["foto_mes"].isin(f["train"]).to_numpy(),
                          (base["foto_mes"] == f["validacion"]).to_numpy())
            for f in folds
        }

    def firma(self, nombre, features, tasa_continua=1.0):
        for c in features:
            if c not in self.huellas:
                self.huellas[c] = huella_columna(self.base[c])
        contenido = json.dumps(
            {"features": [(f, self.huellas[f]) for f in features],
             "params": PARAMS_BASE, "iter": ITERACIONES, "tasa": tasa_continua},
            sort_keys=True, default=str)
        return hashlib.sha1(contenido.encode()).hexdigest()

    def medir(self, nombre, features, semillas, tasa_continua=1.0):
        firma = self.firma(nombre, features, tasa_continua)
        filas = []
        for fold, (mt, mv) in self.mascaras.items():
            X = self.base[list(features)]
            X_tr = X.loc[mt].to_numpy(dtype=np.float32)
            X_va = X.loc[mv].to_numpy(dtype=np.float32)
            y_tr_full = self.y[mt]
            clase_tr = self.clase[mt]
            gan = ganancia_por_fila(self.clase[mv])

            for s in semillas:
                clave = (firma, fold, int(s))
                if clave in self.cache:
                    filas.append(self.cache[clave]); continue

                X_tr_s, y_tr_s = X_tr, y_tr_full
                if tasa_continua < 1.0:
                    # Se submuestrea SOLO CONTINUA, nunca BAJA+2, y nunca en
                    # validación: la ganancia tiene que medirse en la población real.
                    rng = np.random.default_rng(s)
                    keep = ((clase_tr != "CONTINUA")
                            | (rng.random(len(y_tr_full)) < tasa_continua))
                    X_tr_s, y_tr_s = X_tr[keep], y_tr_full[keep]

                t0 = time.time()
                modelo = lgb.train(
                    {**PARAMS_BASE, "seed": s, "bagging_seed": s,
                     "feature_fraction_seed": s, "data_random_seed": s},
                    lgb.Dataset(X_tr_s, label=y_tr_s),
                    num_boost_round=ITERACIONES)
                p = corregir_probabilidad(modelo.predict(X_va), tasa_continua)
                np.save(self.carpeta / "pred" / f"{firma[:16]}_{fold}_{s}.npy",
                        p.astype(np.float32))
                estim = p >= umbral
                r = {"firma": firma, "brazo": nombre, "fold": fold, "semilla": int(s),
                     "ganancia": float(gan[estim].sum()),
                     "estimulados": int(estim.sum()), "segundos": round(time.time()-t0, 1)}
                self.cache[clave] = r
                pd.DataFrame([r]).to_csv(self.ruta, mode="a", index=False,
                                         header=not self.ruta.is_file())
                filas.append(r)
        return pd.DataFrame(filas)

    def comparar(self, inc, cand, semillas=None):
        """Comparación apareada con poda secuencial.

        inc y cand son (nombre, features). Devuelve el veredicto por fold.
        """
        semillas = list(semillas or semillas_medicion)
        n = min(SEMILLAS_PODA, len(semillas))
        while True:
            m_i = self.medir(*inc, semillas[:n])
            m_c = self.medir(*cand, semillas[:n])
            por_fold = {}
            for f in self.mascaras:
                a = m_i[m_i.fold == f].set_index("semilla")["ganancia"]
                b = m_c[m_c.fold == f].set_index("semilla")["ganancia"]
                d = (b - a.reindex(b.index)).to_numpy()
                media_inc = float(a.mean())
                se = float(d.std(ddof=1)/np.sqrt(len(d))) if len(d) > 1 else np.inf
                por_fold[f] = {
                    "n": len(d), "media_inc": media_inc, "media_cand": float(b.mean()),
                    "delta": float(d.mean()),
                    "delta_rel": float(d.mean()/max(abs(media_inc), 1.0)),
                    "cota_sup_90": float(d.mean() + stats.t.ppf(0.95, max(len(d)-1,1))*se),
                    "p_mejor": p_wilcoxon(d, "greater"),
                    "p_peor": p_wilcoxon(d, "less"),
                    "a_favor": int((d > 0).sum()),
                }
            quedan = n < len(semillas)
            minimo = {f: MEJORA_MINIMA_REL*abs(por_fold[f]["media_inc"])
                      for f in por_fold}
            # PODA: ya viene perdiendo con evidencia, o la cota superior no
            # llega a la mejora mínima ni en el mejor caso.
            peor = [f for f in por_fold if por_fold[f]["p_peor"] < ALPHA_PODA]
            corto = [f for f in por_fold if por_fold[f]["cota_sup_90"] < minimo[f]]
            if quedan and (peor or corto):
                motivo = (f"podado con {n} semillas: pierde con evidencia en {peor}"
                          if peor else
                          f"podado con {n} semillas: ni en el mejor caso llega "
                          f"a la mejora mínima en {corto}")
                return {"aceptado": False, "motivo": motivo, "por_fold": por_fold}
            if quedan:
                n = len(semillas); continue
            ok = all(por_fold[f]["p_mejor"] < ALPHA and por_fold[f]["delta"] >= minimo[f]
                     for f in por_fold)
            return {"aceptado": ok, "por_fold": por_fold,
                    "motivo": "gana en los dos folds" if ok
                              else "no gana en los dos folds con significancia y mejora mínima"}


def resumen(veredicto):
    partes = [f"{f}: {r['delta']/1e6:+.2f}M ({r['delta_rel']:+.2%}) "
              f"p={r['p_mejor']:.3f} {r['a_favor']}/{r['n']}"
              for f, r in veredicto["por_fold"].items()]
    return ("ACEPTADO  " if veredicto["aceptado"] else "RECHAZADO ")            + veredicto["motivo"] + " || " + " | ".join(partes)


print("BancoMedicion listo")

BancoMedicion listo


## 9. La primera medición: ¿sirven las 856 features?

La pregunta que todavía no tiene respuesta: **el set con features temporales le gana al
crudo?** Nunca se midió con poder suficiente.

- **Incumbente:** las 144 columnas crudas.
- **Candidato:** las 144 crudas + las temporales.

Ceteris paribus: mismos hiperparámetros, mismas iteraciones, mismas semillas, mismos
folds. Lo único que cambia es el juego de features.

In [34]:
%%time
# Se arma la tabla de modelado uniendo crudo + features, solo con los meses
# que tienen target o que hay que scorear.
con.execute(f"""
    create or replace table modelado as
    select c.*, f.* exclude (numero_de_cliente, foto_mes)
    from competencia_01 c
    join features_temporales f using (numero_de_cliente, foto_mes)
    where c.foto_mes in ({",".join(map(str, meses_target))}) or c.foto_mes = {mes_scoring}
""")

base = con.execute("select * from modelado").df()
base["target_baja2"] = (base["clase_ternaria"] == "BAJA+2").astype("int8")

features_temporales_nombres = [
    c for c in base.columns
    if c not in set(todas) | {"target_baja2"}
]
features_todas = features_crudas + features_temporales_nombres

# LightGBM acepta nulos pero NO infinitos.
import numpy as _np
base[features_todas] = base[features_todas].replace([_np.inf, -_np.inf], _np.nan)

assert not (columnas_prohibidas & set(features_todas)), "hay columnas prohibidas"
print(f"filas: {len(base):,}")
print(f"features crudas     : {len(features_crudas)}")
print(f"features temporales : {len(features_temporales_nombres)}")
print(f"total               : {len(features_todas)}")

filas: 818,713
features crudas     : 144
features temporales : 796
total               : 940
CPU times: user 32.5 s, sys: 12 s, total: 44.5 s
Wall time: 30 s


In [35]:
%%time
banco = BancoMedicion(base)

veredicto = banco.comparar(
    ("crudas",     features_crudas),
    ("con_tiempo", features_todas),
)
print(resumen(veredicto))
print()
pd.DataFrame(veredicto["por_fold"]).T

ACEPTADO  gana en los dos folds || mayo: +6.30M (+2.68%) p=0.001 15/20 | junio: +49.18M (+14.00%) p=0.000 19/20

CPU times: user 7.81 s, sys: 3.89 s, total: 11.7 s
Wall time: 12.4 s


,n,media_inc,media_cand,delta,delta_rel,cota_sup_90,p_mejor,p_peor,a_favor
mayo,20.0,235004000.0,241308375.0,6304375.0,0.026827,9.129104e+06,0.000845,0.999284,15.0
junio,20.0,351331750.0,400514125.0,49182375.0,0.139988,5.753188e+07,0.000002,0.999999,19.0


## Qué sigue

Con esto hay base verificada, target demostrablemente correcto, controles de fuga
corriendo y una fábrica de features que no lee el futuro.

Lo que falta, en orden:

1. **Canaritos.** Agregar columnas de ruido puro y cortar toda feature que no le gane al
   ruido. Es el patrón de `z401` celda 44, usado como criterio de selección. Con cientos
   de features y 0,6 % de positivos, hace falta.
2. **El banco de medición.** 20 semillas × los 2 folds, comparación apareada, Wilcoxon.
   Sin eso no se puede decidir si un bloque de features aporta: el desvío entre semillas
   en junio es de 13,2 M y el protocolo de 5 semillas solo resolvía efectos de ~20 M.
3. **Optuna.** Recién acá, y sobre el set de features **ya fijado**. Es la única parte
   que va a tardar horas, y por eso es la única que conviene sacar a un `.py` y lanzar
   con `nohup`.
4. **Entrenamiento final y entrega.** Train 202103-202106, predicción de 202108, las 5
   semillas declaradas, corte en 0,025.